UTS Data Mining_Mufida Aqila Humaidah_2304020127

A. PERSIAPAN DATA

1. Import Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings('ignore')

2. Load Data

In [ ]:
train_data = pd.read_csv('data_training.csv')
test_data = pd.read_csv('data_testing.csv')

3. Cek Struktur Data

In [ ]:
train_data.info()

In [ ]:
train_data.head()

In [ ]:
test_data.info()

In [ ]:
test_data.head()

B. PEMBERSIHAN DATA

1. Cek Missing Value

In [ ]:
train_data.isnull().sum()

In [ ]:
test_data.isnull().sum()

2. Exploratory Data Analysis (EDA)

a. Distribusi Target

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=train_data, x='quality')
plt.title("Distribusi Quality")
plt.show()

b. Korelasi Fitur

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(train_data.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

3. Preprocessing

a. Pemisahan fitur dan target

In [ ]:
X = train_data.drop(['quality', 'Id'], axis=1)
y = train_data['quality']

b. Split data

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

c. Feature scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

d. Data testing

In [ ]:
X_test = test_data.drop('Id', axis=1)
X_test_scaled = scaler.transform(X_test)

test_ids = test_data['Id']

C. PEMBUATAN MODEL

1. Model Desicion Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train_scaled, y_train)

y_dt = dt.predict(X_val_scaled)
acc_dt = accuracy_score(y_val, y_dt)

print("Akurasi Decision Tree:", acc_dt)

Berdasarkan hasil output, akurasi model Desicion Tree sebesar 56,39% menunjukkan performa yang masih moderat, yang kemungkinan dipengaruhi oleh kompleksitas data atau ketidakseimbangan kelas, sehingga hasil ini dapat dijadikan acuan untuk evaluasi dan optimasi lebih lanjut.


2. Model Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
rf.fit(X_train_scaled, y_train)

y_rf = rf.predict(X_val_scaled)
acc_rf = accuracy_score(y_val, y_rf)

print("Akurasi Random Forest:", acc_rf)

Berdasarkan hasil output, akurasi model Random Forest sebesar 57,55% menunjukkan peningkatan dibanding model sebelumnya karena kemampuannya dalam menangani hubungan non-linear dan mengurangi varians, meskipun performanya masih dipengaruhi oleh ketidakseimbangan kelas pada dataset.


3. Model Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)

y_lr = lr.predict(X_val_scaled)
acc_lr = accuracy_score(y_val, y_lr)

print("Akurasi Logistic Regression:", acc_lr)

Berdasarkan hasil output, akurasi model Logistic Regression sebesar 61,05% merupakan yang tertinggi dibanding model sebelumnya, menunjukkan bahwa pola data cenderung memiliki hubungan linear yang cukup kuat dan model ini lebih stabil dalam menangani varians dibandingkan model berbasis pohon.


4. Model K-Nearest Neighbors (KNN)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_knn = knn.predict(X_val_scaled)
acc_knn = accuracy_score(y_val, y_knn)

print("Akurasi KNN:", acc_knn)

Berdasarkan hasil output, akurasi model KNN sebesar 48,25% menunjukkan bahwa pola data cukup kompleks dan saling tumpang tindih, sehingga metode berbasis jarak kurang efektif dan model kesulitan menggeneralisasi data, kemungkinan karena adanya noise atau fitur yang tidak memisahkan kelas dengan jelas dalam ruang Euclidean.

5. Perbandingan Keempat Model

In [ ]:
hasil = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'Logistic Regression', 'KNN'],
    'Akurasi': [acc_dt, acc_rf, acc_lr, acc_knn]
})

print(hasil)

sns.barplot(data=hasil, x='Model', y='Akurasi')
plt.ylim(0,1)
plt.show()

Berdasarkan hasil output, pada dataset kualitas wine ini model Logistic Regression lebih efektif dalam menangkap hubungan antar fitur dibandingkan model berbasis pohon maupun jarak; namun akurasi sekitar 60% menunjukkan masih adanya tantangan seperti ketidakseimbangan kelas atau overlap fitur, sehingga hasil ini menjadi dasar pemilihan model final untuk prediksi data testing.


6. Evaluasi Model Terbaik (Logistic Regression)

In [ ]:
print(classification_report(y_val, y_lr))

cm = confusion_matrix(y_val, y_lr)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix Logistic Regression")
plt.show()

Berdasarkan hasil output, model sangat bergantung pada jumlah data (support), terlihat dari kelas mayoritas seperti 5 dan 6 yang memiliki skor F1 masing-masing 0,72 dan 0,58, sementara kelas minoritas (3, 4, dan 8) tidak terprediksi sama sekali (skor 0,00); confusion matrix menunjukkan dominasi prediksi benar pada kelas menengah namun masih banyak misklasifikasi pada kelas yang berdekatan, sehingga meskipun akurasi mencapai 61%, nilai macro average yang rendah (0,29) menandakan adanya bias kuat akibat ketidakseimbangan data dan perlunya strategi penyeimbangan untuk meningkatkan performa pada kelas ekstrem.

7. Cross Validation

In [ ]:
cv = cross_val_score(lr, X_train_scaled, y_train, cv=5)
print("CV Mean:", cv.mean())

Berdasarkan hasil output, nilai CV Mean sebesar 59,12% yang mendekati akurasi validasi (61%) menunjukkan model cukup stabil dan tidak mengalami overfitting signifikan, sehingga dapat dianggap andal untuk prediksi, meskipun masih memiliki keterbatasan dalam menangani seluruh variasi kualitas wine.


8. Hyperparameter Tuning

In [ ]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_

print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

Berdasarkan hasil output, Best Params ‘C’: 1 dengan Best Score 59,12% menunjukkan bahwa parameter default sudah optimal untuk dataset ini, menandakan model telah melalui proses tuning yang sistematis dan mencapai performa terbaik yang dapat dihasilkan oleh algoritma tersebut.


In [ ]:
y_best = best_model.predict(X_val_scaled)
print("Akurasi setelah tuning:", accuracy_score(y_val, y_best))

Berdasarkan hasil output, akurasi tetap sebesar 61,04% setelah tuning, yang menunjukkan bahwa model sudah berada pada kondisi optimal dengan parameter default (C=1), sehingga proses tuning tidak memberikan peningkatan signifikan; hal ini juga menandakan model cukup stabil dan tidak mengalami overfitting sebelum digunakan pada data testing akhir.


D. PREDIKSI DATA UJI

1. Prediksi Data Testing

In [ ]:
final_pred = best_model.predict(X_test_scaled)

submission = pd.DataFrame({
    'Id': test_ids,
    'quality': final_pred
})

print(submission.head())

Berdasarkan hasil output, setiap baris pada file tersebut merepresentasikan keputusan akhir model berdasarkan pola yang telah dipelajari saat pelatihan, dengan kolom *quality* berisi prediksi kategori kualitas wine untuk setiap ID; hasil ini menunjukkan implementasi dari model yang telah divalidasi sebelumnya, sehingga file yang dihasilkan dapat digunakan sebagai output akhir untuk menilai kualitas wine secara otomatis dengan tingkat keandalan sesuai akurasi yang diperoleh.


2. Menyimpan CSV

In [ ]:
submission.to_csv('hasilprediksi_127.csv', index=False)

3. Mendownload CSV

In [ ]:
from google.colab import files
files.download('hasilprediksi_127.csv')

4. Verifikasi

In [ ]:
print(submission.columns)
print(len(submission), len(test_data))

Berdasarkan hasil output, format data sudah benar dengan jumlah 286 data yang sesuai, sehingga tidak ditemukan data yang hilang maupun terduplikasi; hal ini menunjukkan pipeline berjalan stabil dan hasil prediksi sudah siap digunakan.


E. KESIMPULAN

Berdasarkan hasil analisis, dapat disimpulkan bahwa Logistic Regression merupakan model terbaik untuk mengklasifikasikan kualitas anggur dibandingkan Decision Tree, Random Forest, dan KNN, dengan akurasi tertinggi sebesar **61,05%**, yang menunjukkan bahwa pola dalam data cenderung linear sehingga lebih sesuai dimodelkan menggunakan pendekatan linear. Meskipun model menunjukkan performa yang cukup baik dan stabil dalam memprediksi data, masih terdapat potensi bias terutama pada kelas kualitas menengah yang disebabkan oleh ketidakseimbangan distribusi kelas dalam dataset, sementara fitur kimia seperti kadar alkohol dan tingkat keasaman diduga memiliki pengaruh yang signifikan terhadap hasil prediksi. Secara keseluruhan, proses preprocessing, pemodelan, serta evaluasi yang dilakukan secara sistematis telah menghasilkan model yang cukup andal dan dapat digunakan sebagai dasar untuk analisis lebih lanjut, meskipun masih terdapat ruang untuk peningkatan performa melalui optimasi model dan penyeimbangan data.
